In [ ]:
import json
from collections import Counter, deque
from datetime import datetime, timedelta

from kafka import KafkaConsumer
from kafka.errors import NoBrokersAvailable

from rich.console import Console
from rich.live import Live
from rich.panel import Panel
from rich.table import Table
from rich.layout import Layout
from rich.text import Text


# =========================
# KONFIGURACJA
# =========================

KAFKA_BOOTSTRAP_SERVERS = "broker:9092"

ORDERS_TOPIC = "restaurant_orders"
PANTRY_TOPIC = "pantry_events"

CONSUMER_GROUP_ID = "restaurant_live_dashboard_group"

# Ile minut danych bierzemy do KPI
KPI_WINDOW_MINUTES = 30
TOP_DISHES_WINDOW_MINUTES = 60

# Norma zamówień na 30 minut.
# Na potrzeby demo ustawiona nisko, żeby było widać statusy.
NORMAL_ORDERS_30_MIN = 10

# Ile ostatnich komunikatów pokazywać w logu
RECENT_LOG_LIMIT = 8


#Startowa liczba pracowników
current_staff_state = {"current_staff": 3}


console = Console()


# =========================
# DANE W PAMIĘCI
# =========================

orders = []
pantry_current = {}
pantry_events = []
recent_logs = deque(maxlen=RECENT_LOG_LIMIT)


# =========================
# KAFKA CONSUMER
# =========================

def create_consumer():
    try:
        consumer = KafkaConsumer(
            ORDERS_TOPIC,
            PANTRY_TOPIC,
            bootstrap_servers=[KAFKA_BOOTSTRAP_SERVERS],
            group_id=CONSUMER_GROUP_ID,
            value_deserializer=lambda value: json.loads(value.decode("utf-8")),
            auto_offset_reset="latest",
            enable_auto_commit=True,
            api_version=(2, 8, 0)
        )

        return consumer

    except NoBrokersAvailable:
        console.print("[red]Nie znaleziono brokera Kafka.[/red]")
        console.print("Sprawdź, czy Kafka działa i czy adres jest poprawny.")
        raise


# =========================
# FUNKCJE CZASU I FILTROWANIA
# =========================

def parse_datetime(value):
    if value is None:
        return None

    if isinstance(value, datetime):
        return value

    try:
        return datetime.fromisoformat(value)
    except Exception:
        return None


def get_recent_orders(minutes):
    now = datetime.now()
    start_time = now - timedelta(minutes=minutes)

    recent = []

    for order in orders:
        created_at = parse_datetime(order.get("created_at") or order.get("timestamp"))

        if created_at and created_at >= start_time:
            recent.append(order)

    return recent


def is_order_active(order):
    actual_ready_at = parse_datetime(order.get("actual_ready_at"))

    if actual_ready_at is None:
        return False

    if order.get("status") == "cancelled_missing_ingredients":
        return False

    return actual_ready_at > datetime.now()


def is_order_completed(order):
    actual_ready_at = parse_datetime(order.get("actual_ready_at"))

    if actual_ready_at is None:
        return False

    if order.get("status") == "cancelled_missing_ingredients":
        return False

    return actual_ready_at <= datetime.now()


# =========================
# KPI
# =========================

def calculate_kpis():
    recent_30 = get_recent_orders(KPI_WINDOW_MINUTES)
    recent_60 = get_recent_orders(TOP_DISHES_WINDOW_MINUTES)

    last_staff = current_staff_state.get("current_staff", 3)

    if orders:
        for order in reversed(orders):
            if order.get("current_staff") is not None:
                last_staff = order.get("current_staff")
                break

    valid_recent_30 = [
        order for order in recent_30
        if order.get("status") != "cancelled_missing_ingredients"
    ]

    active_orders = [
        order for order in orders
        if is_order_active(order)
    ]

    completed_orders = [
        order for order in orders
        if is_order_completed(order)
    ]

    cancelled_orders = [
        order for order in orders
        if order.get("status") == "cancelled_missing_ingredients"
    ]

    total_orders_30 = len(valid_recent_30)
    total_orders_60 = len([
        order for order in recent_60
        if order.get("status") != "cancelled_missing_ingredients"
    ])

    revenue_30 = sum(float(order.get("order_value") or 0) for order in valid_recent_30)

    prep_times = [
        int(order.get("actual_preparation_seconds") or 0)
        for order in valid_recent_30
        if int(order.get("actual_preparation_seconds") or 0) > 0
    ]

    avg_prep_time = sum(prep_times) / len(prep_times) if prep_times else 0

    delayed_orders = [
        order for order in valid_recent_30
        if order.get("is_delayed") is True or order.get("is_delayed") == 1
    ]

    delayed_share = (
        len(delayed_orders) / len(valid_recent_30) * 100
        if valid_recent_30
        else 0
    )

    top_counter = Counter()

    for order in recent_60:
        if order.get("status") == "cancelled_missing_ingredients":
            continue

        dish_name = order.get("dish_name", "Nieznane danie")
        quantity = int(order.get("quantity") or 1)

        top_counter[dish_name] += quantity

    demand_deviation = (
        (total_orders_30 - NORMAL_ORDERS_30_MIN) / NORMAL_ORDERS_30_MIN * 100
        if NORMAL_ORDERS_30_MIN > 0
        else 0
    )

    return {
        "orders_30": total_orders_30,
        "orders_60": total_orders_60,
        "active_orders": len(active_orders),
        "completed_orders": len(completed_orders),
        "cancelled_orders": len(cancelled_orders),
        "avg_prep_time": avg_prep_time,
        "delayed_share": delayed_share,
        "revenue_30": revenue_30,
        "top_dishes": top_counter.most_common(5),
        "demand_deviation": demand_deviation,
        "recent_30": valid_recent_30,
        "current_staff": last_staff
    }


# =========================
# STATUSY OPERACYJNE
# =========================

def get_demand_status(kpis):
    deviation = kpis["demand_deviation"]

    if deviation <= 20:
        return "ZIELONY", "Normalny poziom popytu", "green"

    if deviation <= 80:
        return "ŻÓŁTY", f"Popyt przekracza normę o {deviation:.0f}%", "yellow"

    return "CZERWONY", f"Popyt przekracza normę o {deviation:.0f}%", "red"


def get_delay_status(kpis):
    delayed_share = kpis["delayed_share"]

    if delayed_share < 20:
        return "ZIELONY", "Udział opóźnień w normie", "green"

    if delayed_share < 50:
        return "ŻÓŁTY", f"Udział opóźnień podwyższony: {delayed_share:.1f}%", "yellow"

    return "CZERWONY", f"Udział opóźnień jest zbyt wysoki: {delayed_share:.1f}%", "red"


def get_pantry_status():
    if not pantry_current:
        return "BRAK DANYCH", "Brak danych o spiżarni", "white"

    critical = []
    warning = []

    # Progi pokazowe — w realnej wersji możecie dopasować do pojemności
    for ingredient, amount in pantry_current.items():
        try:
            amount = int(amount)
        except Exception:
            continue

        if amount <= 5:
            critical.append(ingredient)
        elif amount <= 12:
            warning.append(ingredient)

    if critical:
        return "CZERWONY", f"Krytycznie niski stan: {', '.join(critical[:4])}", "red"

    if warning:
        return "ŻÓŁTY", f"Niski stan: {', '.join(warning[:4])}", "yellow"

    return "ZIELONY", "Stan składników stabilny", "green"


def get_overall_status(kpis):
    demand_status, _, demand_color = get_demand_status(kpis)
    delay_status, _, delay_color = get_delay_status(kpis)
    pantry_status, _, pantry_color = get_pantry_status()

    if "CZERWONY" in [demand_status, delay_status, pantry_status]:
        return "CZERWONY", "RYZYKO OPERACYJNE", "red"

    if "ŻÓŁTY" in [demand_status, delay_status, pantry_status]:
        return "ŻÓŁTY", "PODWYŻSZONE RYZYKO", "yellow"

    return "ZIELONY", "SYTUACJA STABILNA", "green"


# =========================
# BUDOWANIE WIDOKU
# =========================

def build_kpi_table(kpis):
    table = Table(title="📊 KPI", expand=True)

    table.add_column("Wskaźnik", style="cyan")
    table.add_column("Wartość", justify="right", style="bold")

    table.add_row("Zamówienia ostatnie 30 min", str(kpis["orders_30"]))
    table.add_row("Zamówienia ostatnie 60 min", str(kpis["orders_60"]))
    table.add_row("Aktywne zamówienia", str(kpis["active_orders"]))
    table.add_row("Zakończone zamówienia", str(kpis["completed_orders"]))
    table.add_row("Anulowane — brak składników", str(kpis["cancelled_orders"]))
    table.add_row("Średni czas przygotowania", f"{kpis['avg_prep_time']:.1f} s")
    table.add_row("Udział opóźnień", f"{kpis['delayed_share']:.1f}%")
    table.add_row("Przychód ostatnie 30 min", f"{kpis['revenue_30']:.2f} PLN")
    table.add_row("Odchylenie od normy", f"{kpis['demand_deviation']:.0f}%")
    table.add_row("Aktualna liczba pracowników", str(kpis["current_staff"]))

    return table


def build_top_dishes_table(kpis):
    table = Table(title="🔥 Top 5 dań ostatnie 60 min", expand=True)

    table.add_column("Danie", style="cyan")
    table.add_column("Liczba", justify="right", style="bold")

    if not kpis["top_dishes"]:
        table.add_row("Brak danych", "-")
        return table

    for dish, count in kpis["top_dishes"]:
        table.add_row(dish, str(count))

    return table


def build_pantry_table():
    table = Table(title="🥫 Aktualny stan spiżarni", expand=True)

    table.add_column("Składnik", style="cyan")
    table.add_column("Stan", justify="right", style="bold")
    table.add_column("Status", justify="center")

    if not pantry_current:
        table.add_row("Brak danych", "-", "-")
        return table

    for ingredient, amount in sorted(pantry_current.items()):
        try:
            amount_int = int(amount)
        except Exception:
            amount_int = 0

        if amount_int <= 5:
            status = "[red]KRYTYCZNY[/red]"
        elif amount_int <= 12:
            status = "[yellow]NISKI[/yellow]"
        else:
            status = "[green]OK[/green]"

        table.add_row(ingredient, str(amount_int), status)

    return table


def build_recent_logs_table():
    table = Table(title="🧾 Ostatnie zdarzenia", expand=True)

    table.add_column("Czas", style="cyan")
    table.add_column("Zdarzenie")

    if not recent_logs:
        table.add_row("-", "Brak zdarzeń")
        return table

    for log in list(recent_logs)[-RECENT_LOG_LIMIT:]:
        table.add_row(log["time"], log["text"])

    return table


def build_status_panel(kpis):
    demand_status, demand_desc, demand_color = get_demand_status(kpis)
    delay_status, delay_desc, delay_color = get_delay_status(kpis)
    pantry_status, pantry_desc, pantry_color = get_pantry_status()
    overall_status, overall_desc, overall_color = get_overall_status(kpis)

    text = Text()

    text.append("⚠️ ALERTY OPERACYJNE\n", style="bold yellow")

    text.append("Popyt: ", style="bold")
    text.append(f"{demand_status}", style=f"bold {demand_color}")
    text.append(f" — {demand_desc}\n")

    text.append("Opóźnienia: ", style="bold")
    text.append(f"{delay_status}", style=f"bold {delay_color}")
    text.append(f" — {delay_desc}\n")

    text.append("Spiżarnia: ", style="bold")
    text.append(f"{pantry_status}", style=f"bold {pantry_color}")
    text.append(f" — {pantry_desc}\n\n")

    text.append("⚙ STATUS OGÓLNY: ", style="bold")
    text.append(f"{overall_status}", style=f"bold {overall_color}")
    text.append(f" — {overall_desc}", style=f"bold {overall_color}")

    return Panel(text, title="Status operacyjny", border_style=overall_color)


def build_recommendations_panel(kpis):
    active = kpis["active_orders"]
    delayed = kpis["delayed_share"]

    recommended_staff = max(1, round(active / 3))

    recommendations = []

    if active >= 10:
        recommendations.append("Zwiększyć obsadę kuchni o 2 osoby.")
    elif active >= 5:
        recommendations.append("Rozważyć zwiększenie obsady kuchni o 1 osobę.")
    else:
        recommendations.append("Utrzymać obecną obsadę.")

    if delayed >= 50:
        recommendations.append("Wydłużyć rekomendowany czas dostawy do 55–70 min.")
    elif delayed >= 20:
        recommendations.append("Wydłużyć rekomendowany czas dostawy do 40–50 min.")
    else:
        recommendations.append("Pozostawić standardowy czas dostawy 30–40 min.")

    pantry_status, pantry_desc, _ = get_pantry_status()

    if pantry_status in ["ŻÓŁTY", "CZERWONY"]:
        recommendations.append(f"Sprawdzić zapasy składników: {pantry_desc}.")

    text = Text()
    text.append(f"Rekomendowana obsada: {recommended_staff} osób\n", style="bold cyan")

    for rec in recommendations:
        text.append(f"• {rec}\n")

    return Panel(text, title="🧠 Rekomendacje", border_style="cyan")


def build_dashboard():
    kpis = calculate_kpis()

    layout = Layout()

    layout.split_column(
        Layout(name="header", size=8),
        Layout(name="main"),
        Layout(name="bottom", size=12)
    )

    layout["main"].split_row(
        Layout(name="left"),
        Layout(name="right")
    )

    layout["left"].split_column(
        Layout(build_kpi_table(kpis)),
        Layout(build_top_dishes_table(kpis))
    )

    layout["right"].split_column(
        Layout(build_pantry_table()),
        Layout(build_recommendations_panel(kpis))
    )

    layout["header"].update(build_status_panel(kpis))
    layout["bottom"].update(build_recent_logs_table())

    return layout


# =========================
# OBSŁUGA WIADOMOŚCI
# =========================

def handle_order(order):
    orders.append(order)

    order_id = order.get("order_id", "-")
    dish = order.get("dish_name", "-")
    quantity = order.get("quantity", 1)
    value = float(order.get("order_value") or 0)
    status = order.get("status", "-")

    recent_logs.append({
        "time": datetime.now().strftime("%H:%M:%S"),
        "text": f"Odebrano {order_id} | {dish} x{quantity} | {value:.2f} PLN | status={status}"
    })

    pantry_after_order = order.get("pantry_after_order", {})

    if pantry_after_order:
        pantry_current.clear()
        pantry_current.update(pantry_after_order)


def handle_pantry_event(event):
    # =========================
    # 1. Obsługa zmiany liczby pracowników
    # =========================
    if event.get("event_type") == "staff_change":
        current_staff_state["current_staff"] = event.get("current_staff", 3)

        recent_logs.append({
            "time": datetime.now().strftime("%H:%M:%S"),
            "text": (
                f"Zmieniono obsadę: "
                f"{event.get('previous_staff')} → {event.get('current_staff')} pracowników"
            )
        })

        return

    # =========================
    # 2. Obsługa uzupełnienia spiżarni
    # =========================
    pantry_after_restock = event.get("pantry_after_restock", {})

    if pantry_after_restock:
        pantry_current.clear()
        pantry_current.update(pantry_after_restock)

    restocked_items = event.get("restocked_items", {})

    added = {
        ingredient: amount
        for ingredient, amount in restocked_items.items()
        if amount > 0
    }

    recent_logs.append({
        "time": datetime.now().strftime("%H:%M:%S"),
        "text": f"Uzupełniono spiżarnię: {added}"
    })


# =========================
# MAIN
# =========================

def main():
    consumer = create_consumer()

    console.print("[bold green]Live dashboard uruchomiony.[/bold green]")
    console.print(f"Nasłuchiwanie topiców: [cyan]{ORDERS_TOPIC}[/cyan], [cyan]{PANTRY_TOPIC}[/cyan]")

    with Live(build_dashboard(), refresh_per_second=2, console=console) as live:
        try:
            for message in consumer:
                topic = message.topic
                data = message.value

                if topic == ORDERS_TOPIC:
                    handle_order(data)

                elif topic == PANTRY_TOPIC:
                    handle_pantry_event(data)

                live.update(build_dashboard())

        except KeyboardInterrupt:
            console.print("[yellow]Consumer/dashboard zatrzymany przez użytkownika.[/yellow]")

        finally:
            consumer.close()
            console.print("[green]Połączenie z Kafka zostało zamknięte.[/green]")


if __name__ == "__main__":
    main()

Live dashboard uruchomiony.

Nasłuchiwanie topiców: restaurant_orders, pantry_events

Output()